# PNKLN Multi-Agent COR Template
## AiYouJR Governance Framework - Vertex AI Workbench

**Objective**: Coordinate WhiteCastle, BrownSnow, and OrangeCreek agents for rust_scriptbots/Bevy integration

**Governance**: ATP 5-19 compliant with PRB risk stratification

**Communication**: GCS-backed Agent Mail system

---

## Quick Navigation
1. [Configuration](#1-configuration)
2. [Agent Registry](#2-agent-registry)
3. [Initialize Agents](#3-initialize-agents)
4. [Task Execution](#4-task-execution)
5. [Monitoring & Validation](#5-monitoring-validation)
6. [Bevy Integration](#6-bevy-integration)

## 1. Configuration
Set your GCP project and authentication

In [ ]:
# REQUIRED: Set your GCP Project ID
PROJECT_ID = "your-project-id"  # EDIT THIS
REGION = "us-central1"
AGENT_BUCKET = f"{PROJECT_ID}-pnkln-agents"

# Verify configuration
print(f"Project ID: {PROJECT_ID}")
print(f"Region: {REGION}")
print(f"Agent Bucket: gs://{AGENT_BUCKET}")

# Set environment
import os

os.environ["GCP_PROJECT_ID"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

In [ ]:
# Install dependencies
!pip install -q claude-agent-sdk anthropic google-cloud-storage

## 2. Agent Registry
Define the three core agents with AiYouJR governance

In [ ]:
from dataclasses import dataclass
from enum import Enum


class RiskLevel(Enum):
    """PRB Risk Stratification Levels"""

    RA_1 = "Routine Administrative"  # Auto-approve
    RA_2 = "Low Risk"  # Auto-approve with logging
    RA_3 = "Medium Risk"  # Requires review
    RA_4 = "High Risk"  # Requires human approval (ATP 5-19 gate)
    RA_5 = "Critical"  # Requires multi-party approval


@dataclass
class AgentConfig:
    name: str
    codename: str
    role: str
    capabilities: list[str]
    risk_level: RiskLevel
    coverage_target: float  # Minimum test coverage required


# Agent Registry
AGENTS = {
    "whitecastle": AgentConfig(
        name="WhiteCastle",
        codename="WC-01",
        role="Architecture & Planning",
        capabilities=[
            "System architecture design",
            "Bevy ECS integration planning",
            "Dependency management",
            "Risk assessment (PRB framework)",
        ],
        risk_level=RiskLevel.RA_3,
        coverage_target=0.95,
    ),
    "brownsnow": AgentConfig(
        name="BrownSnow",
        codename="BS-02",
        role="Implementation & Integration",
        capabilities=[
            "Rust code generation",
            "Bevy plugin development",
            "FastAPI endpoint creation",
            "Claude Agent SDK integration",
        ],
        risk_level=RiskLevel.RA_2,
        coverage_target=0.98,
    ),
    "orangecreek": AgentConfig(
        name="OrangeCreek",
        codename="OC-03",
        role="Validation & Quality Assurance",
        capabilities=[
            "Test suite generation (Judge 6)",
            "Coverage enforcement (98% minimum)",
            "Security validation",
            "ATP 5-19 compliance gating",
        ],
        risk_level=RiskLevel.RA_4,  # Requires human approval
        coverage_target=1.0,  # OrangeCreek must be perfect
    ),
}

# Display registry
for _agent_id, config in AGENTS.items():
    print(f"\n{config.codename} - {config.name}")
    print(f"  Role: {config.role}")
    print(f"  Risk Level: {config.risk_level.value}")
    print(f"  Coverage Target: {config.coverage_target * 100}%")
    print("  Capabilities:")
    for cap in config.capabilities:
        print(f"    - {cap}")

## 3. Initialize Agents
Set up Claude Agent SDK instances with GCS-backed communication

In [ ]:
import json
from datetime import datetime

from google.cloud import storage


class AgentMailSystem:
    """GCS-backed inter-agent communication"""

    def __init__(self, bucket_name: str):
        self.client = storage.Client()
        self.bucket = self.client.bucket(bucket_name)
        self.mail_prefix = "agent-mail/"

    def send_message(self, from_agent: str, to_agent: str, message: dict):
        """Send message via GCS"""
        timestamp = datetime.utcnow().isoformat()
        message_id = f"{from_agent}-to-{to_agent}-{timestamp}"

        envelope = {
            "message_id": message_id,
            "from": from_agent,
            "to": to_agent,
            "timestamp": timestamp,
            "payload": message,
        }

        blob_path = f"{self.mail_prefix}{to_agent}/inbox/{message_id}.json"
        blob = self.bucket.blob(blob_path)
        blob.upload_from_string(json.dumps(envelope, indent=2))

        print(f"📧 {from_agent} → {to_agent}: {message.get('subject', 'No subject')}")
        return message_id

    def read_inbox(self, agent: str, mark_read=True):
        """Read messages from inbox"""
        inbox_prefix = f"{self.mail_prefix}{agent}/inbox/"
        blobs = self.bucket.list_blobs(prefix=inbox_prefix)

        messages = []
        for blob in blobs:
            if blob.name.endswith(".json"):
                content = blob.download_as_string()
                message = json.loads(content)
                messages.append(message)

                if mark_read:
                    # Move to archive
                    archive_path = blob.name.replace("/inbox/", "/archive/")
                    self.bucket.copy_blob(blob, self.bucket, archive_path)
                    blob.delete()

        return messages


# Initialize Agent Mail
agent_mail = AgentMailSystem(AGENT_BUCKET)
print(f"✅ Agent Mail System initialized on gs://{AGENT_BUCKET}")

In [ ]:
import anthropic


class AiYouAgent:
    """Base agent with AiYouJR governance"""

    def __init__(self, config: AgentConfig, mail_system: AgentMailSystem, api_key: str):
        self.config = config
        self.mail = mail_system
        self.client = anthropic.Anthropic(api_key=api_key)
        self.task_log = []

    async def execute_task(self, task: dict) -> dict:
        """Execute task with governance checks"""
        print(f"\n🤖 {self.config.codename} executing: {task.get('description', 'Unknown task')}")

        # Risk assessment
        if self.config.risk_level == RiskLevel.RA_4:
            print(f"⚠️  Risk Level {self.config.risk_level.value} - Requires human approval")
            approval = input("Approve this task? (yes/no): ")
            if approval.lower() != "yes":
                return {"status": "rejected", "reason": "Human disapproval"}

        # Execute via Claude
        f"""
You are {self.config.name} ({self.config.codename}), an AI agent specialized in:
{chr(10).join("- " + cap for cap in self.config.capabilities)}

Role: {self.config.role}
Risk Level: {self.config.risk_level.value}
Coverage Target: {self.config.coverage_target * 100}%

You must follow AiYouJR governance protocols and ATP 5-19 compliance.
"""

        # Log task
        self.task_log.append(
            {"timestamp": datetime.utcnow().isoformat(), "task": task, "status": "in_progress"}
        )

        # Simulate task execution (replace with actual Claude SDK call)
        result = {
            "status": "completed",
            "agent": self.config.codename,
            "task": task,
            "output": f"Task completed by {self.config.name}",
        }

        self.task_log[-1]["status"] = "completed"
        self.task_log[-1]["result"] = result

        return result

    def send_to(self, recipient: str, message: dict):
        """Send message to another agent"""
        return self.mail.send_message(self.config.codename, recipient, message)

    def check_inbox(self):
        """Check for incoming messages"""
        return self.mail.read_inbox(self.config.codename)


# Note: You'll need to set ANTHROPIC_API_KEY environment variable
# For now, we'll use a placeholder
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "sk-ant-...")

# Initialize agents
agents = {agent_id: AiYouAgent(config, agent_mail, API_KEY) for agent_id, config in AGENTS.items()}

print(f"\n✅ {len(agents)} agents initialized")
for _agent_id, agent in agents.items():
    print(f"   - {agent.config.codename}: {agent.config.name}")

## 4. Task Execution
Example: Bevy integration workflow

In [ ]:
# Example workflow: Integrate Bevy with Claude Agent SDK
async def bevy_integration_workflow():
    """
    Coordinate agents for Bevy integration

    Phase 1: WhiteCastle (Planning)
    Phase 2: BrownSnow (Implementation)
    Phase 3: OrangeCreek (Validation)
    """

    print("=" * 70)
    print("BEVY INTEGRATION WORKFLOW - AiYouJR GOVERNANCE")
    print("=" * 70)

    # Phase 1: WhiteCastle plans the architecture
    print("\n📋 PHASE 1: Architecture Planning (WhiteCastle)")
    planning_task = {
        "description": "Design Bevy ECS integration with Claude Agent SDK",
        "requirements": [
            "Plugin architecture for rust_scriptbots",
            "Agent-to-Bevy communication protocol",
            "State synchronization design",
        ],
    }

    plan_result = await agents["whitecastle"].execute_task(planning_task)

    # WhiteCastle sends plan to BrownSnow
    agents["whitecastle"].send_to(
        "BS-02",
        {
            "subject": "Bevy Integration Architecture",
            "plan": plan_result,
            "action_required": "Implement plugin system",
        },
    )

    # Phase 2: BrownSnow implements
    print("\n🛠️  PHASE 2: Implementation (BrownSnow)")

    # BrownSnow checks inbox
    messages = agents["brownsnow"].check_inbox()
    print(f"📬 BrownSnow received {len(messages)} message(s)")

    implementation_task = {
        "description": "Implement Bevy plugin for agent integration",
        "based_on": plan_result,
    }

    impl_result = await agents["brownsnow"].execute_task(implementation_task)

    # BrownSnow sends to OrangeCreek for validation
    agents["brownsnow"].send_to(
        "OC-03",
        {
            "subject": "Bevy Plugin Ready for Validation",
            "implementation": impl_result,
            "action_required": "Run Judge 6 validation",
        },
    )

    # Phase 3: OrangeCreek validates (Judge 6)
    print("\n✅ PHASE 3: Validation (OrangeCreek - Judge 6)")

    messages = agents["orangecreek"].check_inbox()
    print(f"📬 OrangeCreek received {len(messages)} message(s)")

    validation_task = {
        "description": "Validate Bevy integration with 98% coverage requirement",
        "coverage_target": 0.98,
        "implementation": impl_result,
    }

    validation_result = await agents["orangecreek"].execute_task(validation_task)

    print("\n" + "=" * 70)
    print("WORKFLOW COMPLETE")
    print("=" * 70)

    return {"plan": plan_result, "implementation": impl_result, "validation": validation_result}


# Execute workflow
workflow_result = await bevy_integration_workflow()
print("\n✅ Bevy integration workflow completed successfully")

## 5. Monitoring & Validation
Judge 6 enforcement and coverage reporting

In [ ]:
class Cor.Claude_Code_6Validator:
    """Enforce 98% test coverage requirement"""

    MINIMUM_COVERAGE = 0.98

    @staticmethod
    def validate_coverage(coverage_data: dict) -> dict:
        """
        Validate test coverage against 98% threshold

        Args:
            coverage_data: Dict with 'lines_covered' and 'lines_total'

        Returns:
            Validation result with pass/fail status
        """
        lines_covered = coverage_data.get("lines_covered", 0)
        lines_total = coverage_data.get("lines_total", 0)

        if lines_total == 0:
            return {"status": "fail", "reason": "No code to test", "coverage": 0.0}

        coverage = lines_covered / lines_total
        passed = coverage >= Cor.Claude_Code_6Validator.MINIMUM_COVERAGE

        result = {
            "status": "pass" if passed else "fail",
            "coverage": coverage,
            "threshold": Cor.Claude_Code_6Validator.MINIMUM_COVERAGE,
            "lines_covered": lines_covered,
            "lines_total": lines_total,
        }

        if not passed:
            result["reason"] = (
                f"Coverage {coverage:.2%} below threshold {Cor.Claude_Code_6Validator.MINIMUM_COVERAGE:.2%}"
            )

        return result

    @staticmethod
    def generate_report(validation_results: list[dict]) -> str:
        """Generate coverage report"""
        report = ["\n" + "=" * 70]
        report.append("JUDGE #6 COVERAGE REPORT")
        report.append("=" * 70)
        report.append(f"Minimum Required: {Cor.Claude_Code_6Validator.MINIMUM_COVERAGE:.2%}\n")

        for i, result in enumerate(validation_results, 1):
            status_symbol = "✅" if result["status"] == "pass" else "❌"
            report.append(f"{status_symbol} Test {i}: {result.get('coverage', 0):.2%} coverage")
            if result["status"] == "fail":
                report.append(f"   Reason: {result.get('reason', 'Unknown')}")

        report.append("\n" + "=" * 70)
        return "\n".join(report)


# Example validation
sample_coverage = {"lines_covered": 980, "lines_total": 1000}

validation = Cor.Claude_Code_6Validator.validate_coverage(sample_coverage)
print(Cor.Claude_Code_6Validator.generate_report([validation]))

## 6. Bevy Integration
Example Rust code generation for Bevy plugin

In [ ]:
# Generate Bevy plugin template
BEVY_PLUGIN_TEMPLATE = """
// Auto-generated by BrownSnow (BS-02)
// AiYouJR Governance Framework

use bevy::prelude::*;
use serde::{Deserialize, Serialize};

#[derive(Debug, Clone, Serialize, Deserialize)]
pub struct AgentCommand {
    pub agent_id: String,
    pub command_type: CommandType,
    pub payload: serde_json::Value,
}

#[derive(Debug, Clone, Serialize, Deserialize)]
pub enum CommandType {
    SpawnEntity,
    UpdateComponent,
    QueryState,
    ExecuteSystem,
}

pub struct ClaudeAgentPlugin;

impl Plugin for ClaudeAgentPlugin {
    fn build(&self, app: &mut App) {
        app
            .add_event::<AgentCommand>()
            .add_systems(Update, (
                process_agent_commands,
                sync_agent_state,
            ));
    }
}

fn process_agent_commands(
    mut commands: Commands,
    mut agent_events: EventReader<AgentCommand>,
) {
    for event in agent_events.read() {
        match event.command_type {
            CommandType::SpawnEntity => {
                info!("Agent {} spawning entity", event.agent_id);
                // Handle entity spawn
            }
            CommandType::UpdateComponent => {
                info!("Agent {} updating component", event.agent_id);
                // Handle component update
            }
            CommandType::QueryState => {
                info!("Agent {} querying state", event.agent_id);
                // Handle state query
            }
            CommandType::ExecuteSystem => {
                info!("Agent {} executing system", event.agent_id);
                // Handle system execution
            }
        }
    }
}

fn sync_agent_state() {
    // Sync state to GCS Agent Mail
    // TODO: Implement GCS integration
}

#[cfg(test)]
mod tests {
    use super::*;

    #[test]
    fn test_agent_command_serialization() {
        let cmd = AgentCommand {
            agent_id: "WC-01".to_string(),
            command_type: CommandType::SpawnEntity,
            payload: serde_json::json!({"entity": "player"}),
        };

        let serialized = serde_json::to_string(&cmd).unwrap();
        let deserialized: AgentCommand = serde_json::from_str(&serialized).unwrap();

        assert_eq!(cmd.agent_id, deserialized.agent_id);
    }

    // Judge 6: Ensure 98% coverage - add more tests here
}
"""

# Save to file (would be done by BrownSnow agent)
print("📝 Bevy Plugin Template (claude_agent_plugin.rs):")
print("=" * 70)
print(BEVY_PLUGIN_TEMPLATE)
print("=" * 70)
print("\n✅ Template ready for rust_scriptbots integration")

## Summary

This notebook provides:
1. ✅ Agent registry with AiYouJR governance
2. ✅ GCS-backed Agent Mail communication
3. ✅ ATP 5-19 compliant risk gating
4. ✅ Judge 6 coverage enforcement (98%)
5. ✅ Bevy plugin template generation

## Next Steps
1. Set `ANTHROPIC_API_KEY` environment variable
2. Update `PROJECT_ID` in Cell 2
3. Clone `rust_scriptbots` repository
4. Run all cells to activate agents
5. Monitor `gs://{PROJECT_ID}-pnkln-agents/agent-mail/`

## Resources
- [AGENTS.md](./AGENTS.md) - Full agent registry
- [AIYOUJR_DOCTRINE.md](./AIYOUJR_DOCTRINE.md) - Governance framework
- [PLAN_TO_INTEGRATE_BEVY_ENGINE.md](./PLAN_TO_INTEGRATE_BEVY_ENGINE.md) - Integration roadmap